# Single admin unit validation

This notebook validates the package against a full single
administrative unit EbolaSim run, not the tiny synthetic demo data.
The reference scenario contains 900,000 individuals and was run
with the public `mrc-ide/ebolasim_public` codebase, the associated
input parameter files, demography/population input, and network
input from `ignore/Ebola2.zip`.

The zip is intentionally not committed. The committed evidence in
`docs/vignettes/sadv/evidence` includes command metadata, a
comparison summary, and selected reference/generated outputs for
paramsets 1 and 2. That lets the docs and tests compare package
output against the reference run without putting the full private
bundle in git.


In [ ]:
import csv
import importlib.util
import json
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

from ebolasim import read_results, resolve_executable
from ebolasim_tools.command import build_command_plan
from ebolasim_tools.manifest import (
    ManifestInputs,
    ManifestModelArgs,
    ManifestOutputs,
    ManifestSource,
    RunManifest,
)
from ebolasim_tools.run import run_model


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("could not locate repository root")

ROOT = find_repo_root()
EVIDENCE = ROOT / "docs/vignettes/sadv/evidence"
REPLAY_PATH = ROOT / "docs/vignettes/sadv/replay_sadv.py"
spec = importlib.util.spec_from_file_location("sadv_replay_notebook", REPLAY_PATH)
replay = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = replay
spec.loader.exec_module(replay)

summary = json.loads((EVIDENCE / "comparison_summary.json").read_text(encoding="utf-8"))
{
    "overall_status": summary["overall_status"],
    "paramsets": summary["paramsets_requested"],
    "reference_files_checked": sum(
        item["comparison"]["files_expected"]
        for item in summary["paramsets"]
    ),
    "mismatches": sum(
        item["comparison"]["mismatch_count"]
        for item in summary["paramsets"]
    ),
}


The original run is described by Windows batch files in the zip.
The replay reads those files, preserves `/P`, `/PP`, `/D`, `/S`,
`/R`, `/CLP1..13`, the four random seeds, and `OMP_NUM_THREADS`,
and rewrites only `/O` so reference outputs are never overwritten.


In [ ]:
zip_path = ROOT / "ignore/Ebola2.zip"
bundle_root = None
if zip_path.is_file():
    extracted = replay.safe_extract_zip(
        zip_path,
        ROOT / "artifacts/sadv-notebook/extracted",
    )
    bundle_root = replay.find_ebola_dir(extracted)
    print(f"Bundle root: {bundle_root}")
    batch_files = sorted(
        path.relative_to(bundle_root)
        for path in bundle_root.rglob("*.bat")
    )
    print("Batch files:")
    for path in batch_files:
        print(f"  {path}")
else:
    print(
        "ignore/Ebola2.zip is not present in this environment; "
        "using committed evidence."
    )


The two committed paramsets differ only in the reproduction-number
scale passed to `/R`: paramset 1 uses `1.17`, and paramset 2 uses
`1`. Both use the same parameter file placeholders, CLP launch
substitutions, and random seeds.


In [ ]:
command_plans = []
for paramset in (1, 2):
    path = EVIDENCE / "commands" / f"paramset_{paramset}.json"
    command_plans.append(json.loads(path.read_text(encoding="utf-8")))

[
    {
        "paramset": plan["paramset"],
        "threads": plan["environment"]["OMP_NUM_THREADS"],
        "r_scale": next(arg for arg in plan["command"] if arg.startswith("/R:")),
        "first_clp": next(arg for arg in plan["command"] if arg.startswith("/CLP1:")),
        "last_clp": next(arg for arg in plan["command"] if arg.startswith("/CLP13:")),
        "seeds": plan["command"][-4:],
    }
    for plan in command_plans
]


The CLP values replace `#1..#13` placeholders in the parameter
file. They control the operational scenario layered on top of the
fixed demography and network inputs.

| Placeholder | Meaning |
|---|---|
| `#1` | ETU bed capacity values |
| `#2` | Contact tracing capacity values |
| `#3` | Detected cases required to trigger outbreak alert |
| `#4` | Mean detection delay |
| `#5..#10` | Vaccine delivery capacity, stock, and timing values |
| `#11` | Burial capacity values |
| `#12` | Pre-outbreak HCW/FLW vaccination proportion |
| `#13` | Relative susceptibility of vaccinated HCWs/FLWs |


A user-facing package workflow does not need to parse the batch
files directly once these inputs are known. Build a `RunManifest`
that points at the real C-model input files, turn it into a command
plan for inspection, then call `run_model()` and read the outputs.


In [ ]:
PARAMETER_FILE = "Ervebo/Gavi/Data/p_R1.80_NordKivu_HCWring_midAccept.txt"
PREPARAMETER_FILE = "Ervebo/Gavi/Data/preFPNordKivu_HCWring_singleAdUnit.txt"
DENSITY_FILE = "Ervebo/Populations/NordKivu_MSF_LS2018.bin"
NETWORK_FILE = "Ervebo/Populations/NordKivu_MSF_Network_HCW_singleAdUnt.bin"
SEEDS = [98798150, 729101, 17389101, 4797132]

def manifest_for_paramset(paramset, root, output_root):
    plan = next(item for item in command_plans if item["paramset"] == paramset)
    launch_values = plan["launch_values"]
    output_dir = Path(output_root) / f"paramset_{paramset}"
    return RunManifest(
        inputs=ManifestInputs(
            parameter_file=PARAMETER_FILE,
            preparameter_file=PREPARAMETER_FILE,
            density_file=DENSITY_FILE,
            network_file=NETWORK_FILE,
            network_mode="save",
        ),
        outputs=ManifestOutputs(
            output_base=(output_dir / f"paramset_{paramset}").as_posix(),
            output_dir=output_dir.as_posix(),
        ),
        paramset=paramset,
        threads=4,
        model_args=ManifestModelArgs(
            r0_scale=launch_values[1],
            clp={
                index: value
                for index, value in enumerate(launch_values[2:], start=1)
            },
        ),
        seeds=SEEDS,
        source=ManifestSource(
            kind="single_admin_unit_validation",
            bundle_root=Path(root).as_posix(),
        ),
        metadata={"source_bundle": "ignore/Ebola2.zip"},
    )

example_root = bundle_root if bundle_root is not None else ROOT / "Ebola2"
example_manifest = manifest_for_paramset(
    1,
    example_root,
    ROOT / "artifacts/sadv-package-run/generated",
)
example_plan = build_command_plan(
    example_manifest,
    executable=resolve_executable(required=False) or "ebola-spatial-linux",
    root=example_root,
    working_directory=example_root,
    threads=example_manifest.threads,
)

{
    "parameter_file": example_manifest.inputs.parameter_file,
    "density_file": example_manifest.inputs.density_file,
    "network_file": example_manifest.inputs.network_file,
    "r0_scale": example_manifest.model_args.r0_scale,
    "clp_count": len(example_manifest.model_args.clp),
    "seeds": example_manifest.seeds,
    "shell_command": example_plan.shell_command,
}


In [ ]:
if os.environ.get("EBOLASIM_RUN_SADV") == "1" and bundle_root is not None:
    result = run_model(
        example_manifest,
        executable=resolve_executable(),
        root=bundle_root,
        run_dir=ROOT / "artifacts/sadv-package-run/logs/paramset_1",
        timeout=120,
        threads=example_manifest.threads,
    )
    results = read_results(result.output_dir)
    {
        "classification": result.classification,
        "output_files": len(result.output_files),
        "summary": results.summary,
    }
else:
    print(
        "Package-run cell skipped. Set EBOLASIM_RUN_SADV=1 "
        "and provide ignore/Ebola2.zip to execute it."
    )


The full comparison report checks every expected CSV. The subset
below recomputes direct byte/numeric comparisons for committed
reference/generated pairs: realisations 0 and 999, all five output
file types, for paramsets 1 and 2.


In [ ]:
subset_rows = []
for paramset in (1, 2):
    for realisation in replay.SELECTED_REALISATIONS:
        for suffix in replay.OUTPUT_SUFFIXES:
            filename = replay.paramset_output_name(paramset, realisation, suffix)
            reference = (
                EVIDENCE
                / "output_subset/reference"
                / f"paramset_{paramset}"
                / filename
            )
            generated = (
                EVIDENCE
                / "output_subset/generated"
                / f"paramset_{paramset}"
                / filename
            )
            comparison = replay.compare_output_file(
                reference,
                generated,
                tolerance=1e-9,
            )
            subset_rows.append(comparison.to_dict())

Counter(row["status"] for row in subset_rows)


In [ ]:
def read_series(path, x="t", y="incI"):
    with Path(path).open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
    return [float(row[x]) for row in rows], [float(row[y]) for row in rows]

fig, ax = plt.subplots(figsize=(8, 4))
for paramset in (1, 2):
    for kind, style in (("reference", "-"), ("generated", "--")):
        path = (
            EVIDENCE
            / f"output_subset/{kind}"
            / f"paramset_{paramset}"
            / f"paramset_{paramset}.0.csv"
        )
        t, inc = read_series(path, y="incI")
        ax.plot(t, inc, style, label=f"paramset {paramset} {kind}")

ax.set_xlabel("Day")
ax.set_ylabel("Incident infections")
ax.legend()
fig.tight_layout()
fig


To rerun the full validation locally, provide the zip and opt in to
the slow replay. This is deliberately not a default CI step.


In [ ]:
if os.environ.get("EBOLASIM_RUN_SADV") == "1" and zip_path.is_file():
    exit_code = replay.main([
        "--zip", zip_path.as_posix(),
        "--workdir", (ROOT / "artifacts/sadv-replay").as_posix(),
        "--evidence-dir", EVIDENCE.as_posix(),
        "--paramsets", "1", "2",
        "--threads", "4",
    ])
    print(f"Replay exit code: {exit_code}")
else:
    print(
        "Full replay skipped. Set EBOLASIM_RUN_SADV=1 "
        "and provide ignore/Ebola2.zip."
    )
